[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/chatbot-summarization.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239436-lesson-5-chatbot-w-summarizing-messages-and-memory)

# 具有消息摘要功能的聊天机器人

## 回顾

我们已经学习了如何自定义图状态模式和reducer。
 
我们还展示了许多在图状态中修剪或过滤消息的方法。

## 目标

现在，让我们更进一步！

与其仅仅修剪或过滤消息，我们将展示如何使用LLM来产生对话的运行摘要。
 
这使我们能够保留完整对话的压缩表示，而不是仅仅通过修剪或过滤来删除它。

我们将把这种摘要功能集成到一个简单的聊天机器人中。

并且我们将为这个聊天机器人配备记忆功能，支持长时间运行的对话而不会产生高昂的token成本/延迟。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_openai

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

我们将使用[LangSmith](https://docs.smith.langchain.com/)进行[追踪](https://docs.smith.langchain.com/concepts/tracing)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o",temperature=0)

我们将像之前一样使用`MessagesState`。

除了内置的`messages`键之外，我们现在还将包含一个自定义键（`summary`）。

In [ ]:
from langgraph.graph import MessagesState
class State(MessagesState):
    summary: str

我们将定义一个节点来调用我们的LLM，如果存在摘要，则将其合并到提示中。

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

# 定义调用模型的逻辑
def call_model(state: State):
    
    # 获取摘要（如果存在）
    summary = state.get("summary", "")

    # 如果有摘要，我们添加它
    if summary:
        
        # 将摘要添加到系统消息
        system_message = f"更早对话的摘要: {summary}"

        # 将摘要附加到任何较新的消息
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages)
    return {"messages": response}

我们将定义一个节点来生成摘要。

注意，这里我们将使用`RemoveMessage`在生成摘要后过滤我们的状态。

In [ ]:
def summarize_conversation(state: State):
    
    # 首先，我们获取任何现有的摘要
    summary = state.get("summary", "")

    # 创建我们的摘要提示
    if summary:
        
        # 摘要已经存在
        summary_message = (
            f"这是到目前为止对话的摘要: {summary}\n\n"
            "通过考虑上面的新消息来扩展摘要:"
        )
        
    else:
        summary_message = "创建上述对话的摘要:"

    # 将提示添加到我们的历史记录
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 删除除最近2条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

我们将添加一个条件边，根据对话长度确定是否生成摘要。

In [ ]:
from langgraph.graph import END
from typing_extensions import Literal
# 确定是否结束或摘要对话
def should_continue(state: State) -> Literal ["summarize_conversation",END]:
    
    """返回要执行的下一个节点。"""
    
    messages = state["messages"]
    
    # 如果有超过六条消息，那么我们摘要对话
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否则我们可以直接结束
    return END

## 添加记忆

回想一下，[状态对于单个图执行是临时的](https://github.com/langchain-ai/langgraph/discussions/352#discussioncomment-9291220)。

这限制了我们进行具有中断的多轮对话的能力。

正如在模块1的末尾介绍的，我们可以使用[持久化](https://langchain-ai.github.io/langgraph/how-tos/persistence/)来解决这个问题！
 
LangGraph可以使用检查点保存器在每一步后自动保存图状态。

这个内置的持久化层给我们提供了记忆，允许LangGraph从最后一次状态更新继续进行。

正如我们之前展示的，最容易使用的是`MemorySaver`，这是一个用于图状态的内存键值存储。

我们需要做的就是使用检查点保存器编译图，我们的图就有了记忆！

In [ ]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START

# 定义一个新的图
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 设置入口点为conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 编译
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

## 线程

检查点保存器在每一步将状态保存为检查点。

这些保存的检查点可以分组到对话的`线程`中。

可以将Slack作为类比：不同的频道承载不同的对话。

线程就像Slack频道，捕获状态的分组集合（例如，对话）。

下面，我们使用`configurable`来设置线程ID。

![state.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbadf3b379c2ee621adfd1_chatbot-summarization1.png)

In [ ]:
# 创建一个线程
config = {"configurable": {"thread_id": "1"}}

# 开始对话
input_message = HumanMessage(content="你好！我是Lance")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="我的名字是什么？")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="我喜欢49人队！")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

现在，我们还没有状态摘要，因为我们仍然有<= 6条消息。

这在`should_continue`中设置。

```
    # 如果有超过六条消息，那么我们摘要对话
    if len(messages) > 6:
        return "summarize_conversation"
```

我们可以继续对话，因为我们有线程。

In [ ]:
graph.get_state(config).values.get("summary","")

带有线程ID的`config`允许我们从之前记录的状态继续！

In [ ]:
input_message = HumanMessage(content="我喜欢Nick Bosa，他不是薪水最高的防守球员吗？")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

In [ ]:
graph.get_state(config).values.get("summary","")

## LangSmith

让我们查看追踪！